# Data Cleaning & Feature Engineering
This notebook outlines the systematic process of preparing the dataset for machine learning models. The pipeline includes data inspection, missing values treatment, feature selection, and categorical encoding.

In [1]:
import pandas as pd
import numpy as np


X = pd.read_csv("../data/processed/all_X.csv", index_col="Id")

print(f"Dataset shape initially: {X.shape}")

Dataset shape initially: (2919, 79)


## 1. Initial Data Inspection
Checking for duplicates and analyzing the proportion of missing values across all features to determine our imputation strategy.

In [2]:
print(f"Total duplicated rows: {X.duplicated().sum()}")
X = X.drop_duplicates()

    
missing_stats = X.isna().mean() * 100
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Missing_Ratio (%)': missing_stats
})

missing_df.head(15)

Total duplicated rows: 2


,Missing_Ratio (%)
PoolQC,99.657182
MiscFeature,96.400411
Alley,93.246486
Fence,80.425094
MasVnrType,60.507371
FireplaceQu,48.611587
LotFrontage,16.660953
GarageFinish,5.450806
GarageQual,5.450806
GarageCond,5.450806


## 2. Feature Selection & Dropping Irrelevant Data
Based on the missing values analysis and domain knowledge, we drop features that are either heavily missing (>80%), highly unbalanced (low variance), redundant, or difficult to impute reliably.

In [3]:
#all dropped columns
cols_to_drop = [
    # High missing values (>80%)
    'PoolQC', 'MiscFeature', 'Alley', 'Fence',
    # High missing values & difficult to impute
    'FireplaceQu', 'MasVnrType',
    # Redundant or low-variance
    'Utilities', 'Condition1', 'Condition2', 'RoofMatl', 
    'Functional', 'BldgType', 'GarageCond'
]

X = X.drop(columns=cols_to_drop, errors='ignore')

print(f"Dataset shape after dropping features: {X.shape}")

Dataset shape after dropping features: (2917, 66)


## 3. Handling Missing Values (Imputation & Row Dropping)
For the remaining missing values, we apply logical imputation based on feature relationships. 
Records with inexplicable inconsistencies or negligible missing values are dropped.

In [4]:
# 3.1 Impute numeric features with 0
numeric_impute_cols = ['LotFrontage', 'MasVnrArea', 'BsmtFullBath', 'BsmtHalfBath']
X[numeric_impute_cols] = X[numeric_impute_cols].fillna(0)



# 3.2 Drop rows with impossible basement records
bsmt_cols = ['BsmtQual', 'BsmtCond', 'BsmtFinType1', 'BsmtExposure', 'BsmtFinType2']
mask_bsmt_impossible = X[bsmt_cols].isna().any(axis=1) & (X['TotalBsmtSF'] > 0)
X = X[~mask_bsmt_impossible]



bsmt_area_cols = ['BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF']
X[bsmt_area_cols] = X[bsmt_area_cols].fillna(0)

# 3.3 Clean up Garage features
garage_cols = ['GarageType', 'GarageFinish', 'GarageQual', 'GarageYrBlt']
mask_garage_keep = (X[garage_cols].isna().any(axis=1) & (X['GarageArea'] == 0)) | X[garage_cols].notna().all(axis=1)
X = X[mask_garage_keep]
# Drop any remaining unexplainable NaNs in garage columns
X = X.dropna(subset=garage_cols)

# 3.4 Drop remaining rows with negligible missing values in critical features
negligible_na_cols = ['Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'MSZoning', 'SaleType', 'GarageArea']
X = X.dropna(subset=negligible_na_cols)

print(f"Dataset shape after row cleaning: {X.shape}")

Dataset shape after row cleaning: (2742, 66)


## 4. Feature Encoding
Converting categorical variables into numerical formats suitable for machine learning algorithms.
* **Boolean Encoding:** Yes/No features.
* **Ordinal Encoding:** Categorical features with an inherent order.
* **One-Hot Encoding:** Nominal categorical features without order.

In [5]:
# 4.1 Boolean Encoding
X['CentralAir'] = X['CentralAir'].map({'Y': 1, 'N': 0})

# 4.2 Ordinal Encoding - Defining mappings
quality_map = {np.nan: 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
bsmt_fin_map = {np.nan: 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}

ordinal_mappings = {
    'BsmtQual': quality_map,
    'BsmtCond': quality_map,
    'ExterQual': quality_map,
    'ExterCond': quality_map,
    'HeatingQC': quality_map,
    'KitchenQual': {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}, # No NaNs expected
    'GarageQual': {'NA': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5},
    'BsmtFinType1': bsmt_fin_map,
    'BsmtFinType2': bsmt_fin_map,
    'BsmtExposure': {np.nan: 0, 'No': 1, 'Mn': 2, 'Rec': 3, 'Av': 4, 'Gd': 5},
    'Electrical': {'Mix': 1, 'FuseP': 1, 'FuseF': 2, 'FuseA': 3, 'SBrkr': 4},
    'Street': {'Grvl': 0, 'Pave': 1},
    'PavedDrive': {'N': 0, 'P': 1, 'Y': 2},
    'LotShape': {'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3},
    'LandContour': {'Low': 0, 'HLS': 1, 'Bnk': 2, 'Lvl': 3},
    'LandSlope': {'Sev': 0, 'Mod': 1, 'Gtl': 2},
    'GarageFinish': {'NA': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
}

# Apply ordinal mappings
for col, mapping in ordinal_mappings.items():
    if col in X.columns:
        X[col] = X[col].map(mapping)

# 4.3 One-Hot Encoding
nominal_features = [
    'Exterior2nd', 'Exterior1st', 'MSZoning', 'Neighborhood', 'LotConfig', 
    'RoofStyle', 'Foundation', 'Heating', 'SaleType', 'HouseStyle', 
    'GarageType', 'SaleCondition'
]

X = pd.get_dummies(X, columns=nominal_features, drop_first=True) # drop_first=True helps avoid multicollinearity

# Final check
print(f"Final dataset shape ready for modeling: {X.shape}")
X.info()

Final dataset shape ready for modeling: (2742, 154)
<class 'pandas.core.frame.DataFrame'>
Index: 2742 entries, 1 to 2919
Columns: 154 entries, MSSubClass to SaleCondition_Partial
dtypes: bool(100), float64(11), int64(43)
memory usage: 1.4 MB


Całkowita liczba brakujących wartości (NaN): 0
Kolumny z NaN: []


## 5. Save Clean Data
Saving the final feature matrix for the modeling notebook.

In [6]:
X.to_csv('../data/processed/X_clean.csv')
print('Saved X_clean.csv')

Saved X_clean.csv
